In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/README_KAGGLE_FINAL.md
/kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/checkpoint_after_cell58_FINAL_20260925/KAGGLE_BOOTSTRAP.py
/kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/checkpoint_after_cell58_FINAL_20260925/OLD_AMLC2026_RESUME_BOOTSTRAP.py
/kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/checkpoint_after_cell58_FINAL_20260925/README_KAGGLE.md
/kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/checkpoint_after_cell58_FINAL_20260925/OLD_artifact_manifest.json
/kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/checkpoint_after_cell58_FINAL_20260925/CHECKPOINT_MANIFEST.json
/kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/checkpoint_after_cell58_FINAL_20260925/OLD_README_RESUME.md
/kagg

In [2]:
# ============================================================
# AMLC 2026 — KAGGLE CELL 1
# EXACT CHECKPOINT RESTORE + ENVIRONMENT VERIFICATION
# ============================================================

import os
import gc
import json
import psutil
from pathlib import Path

import polars as pl
import pyarrow.parquet as pq


# ============================================================
# 1) EXACT PACKAGE ROOT
# ============================================================

WORKSPACE_ROOT = Path(
    "/kaggle/input/datasets/tanmayistired/"
    "amlc-2026-final-workspace/"
    "AMLC2026_KAGGLE_FINAL"
)

CHECKPOINT_ROOT = (
    WORKSPACE_ROOT /
    "checkpoint_after_cell58_FINAL_20260925"
)

DATASET_ROOT = (
    WORKSPACE_ROOT /
    "dataset"
)

TRAIN_ROOT = DATASET_ROOT / "train"
TEST_ROOT = DATASET_ROOT / "test"

STATE_ROOT = CHECKPOINT_ROOT / "state"
RUNTIME_ROOT = CHECKPOINT_ROOT / "runtime"
BLOCKING_ROOT = CHECKPOINT_ROOT / "blocking"
FULL_TRAIN_ROOT = CHECKPOINT_ROOT / "full_training"


# ============================================================
# 2) BASIC EXISTENCE CHECK
# ============================================================

print("=" * 72)
print("AMLC 2026 — KAGGLE CHECKPOINT RESTORE")
print("=" * 72)

required_dirs = {
    "WORKSPACE_ROOT": WORKSPACE_ROOT,
    "CHECKPOINT_ROOT": CHECKPOINT_ROOT,
    "DATASET_ROOT": DATASET_ROOT,
    "TRAIN_ROOT": TRAIN_ROOT,
    "TEST_ROOT": TEST_ROOT,
    "STATE_ROOT": STATE_ROOT,
    "RUNTIME_ROOT": RUNTIME_ROOT,
}

for name, path in required_dirs.items():
    print(
        f"{'OK   ' if path.exists() else 'MISS '}{name}: {path}"
    )

missing_dirs = [
    name
    for name, path in required_dirs.items()
    if not path.exists()
]

if missing_dirs:
    raise RuntimeError(
        "Missing required directories:\n" +
        "\n".join(missing_dirs)
    )


# ============================================================
# 3) KEY ARTIFACT PATHS
# ============================================================

CANDIDATE_PATH = (
    STATE_ROOT /
    "eval_candidates_address_rescue.parquet"
)

CANDIDATE_COUNTS_PATH = (
    STATE_ROOT /
    "eval_candidate_counts_address_rescue.parquet"
)

CANDIDATE_RESIDUAL_PATH = (
    STATE_ROOT /
    "eval_candidate_residual_address_rescue.parquet"
)

S1_LOOKUP_PATH = (
    RUNTIME_ROOT /
    "s1_pair_lookup.parquet"
)

S2_NAME_PATH = (
    RUNTIME_ROOT /
    "s2_name_lookup.parquet"
)

S2_ADDRESS_PATH = (
    RUNTIME_ROOT /
    "s2_address_lookup.parquet"
)

S3_NAME_PATH = (
    RUNTIME_ROOT /
    "s3_name_lookup.parquet"
)

S3_ADDRESS_PATH = (
    RUNTIME_ROOT /
    "s3_address_lookup.parquet"
)

GT_PATH = (
    FULL_TRAIN_ROOT /
    "gt.parquet"
)

GT_EDGES_PATH = (
    FULL_TRAIN_ROOT /
    "gt_edges.parquet"
)

CHECKPOINT_MANIFEST = (
    CHECKPOINT_ROOT /
    "CHECKPOINT_MANIFEST.json"
)


# ============================================================
# 4) VERIFY CRITICAL FILES
# ============================================================

critical_files = {
    "candidate_pairs": CANDIDATE_PATH,
    "candidate_counts": CANDIDATE_COUNTS_PATH,
    "candidate_residual": CANDIDATE_RESIDUAL_PATH,
    "s1_lookup": S1_LOOKUP_PATH,
    "s2_name_lookup": S2_NAME_PATH,
    "s2_address_lookup": S2_ADDRESS_PATH,
    "s3_name_lookup": S3_NAME_PATH,
    "s3_address_lookup": S3_ADDRESS_PATH,
    "gt": GT_PATH,
    "gt_edges": GT_EDGES_PATH,
    "manifest": CHECKPOINT_MANIFEST,
}

print("\nCritical artifacts:")

for name, path in critical_files.items():
    print(
        f"{'OK   ' if path.exists() else 'MISS '}{name}: {path.name}"
    )

missing_files = [
    name
    for name, path in critical_files.items()
    if not path.exists()
]

if missing_files:
    raise RuntimeError(
        "Missing critical checkpoint files:\n" +
        "\n".join(missing_files)
    )


# ============================================================
# 5) READ MANIFEST
# ============================================================

with open(
    CHECKPOINT_MANIFEST,
    "r",
    encoding="utf-8"
) as f:
    CHECKPOINT_INFO = json.load(f)

print("\nCheckpoint status:")
print(
    "  status:",
    CHECKPOINT_INFO.get("status")
)

print(
    "  stopping_point:",
    CHECKPOINT_INFO.get("stopping_point")
)

print(
    "  candidate_pairs:",
    f"{CHECKPOINT_INFO.get('candidate_pairs', 0):,}"
)


# ============================================================
# 6) VERIFY CANDIDATE PARQUET WITHOUT LOADING IT
# ============================================================

candidate_meta = pq.ParquetFile(
    str(CANDIDATE_PATH)
).metadata

candidate_rows = candidate_meta.num_rows

print("\nCandidate artifact:")
print(
    "  rows:",
    f"{candidate_rows:,}"
)

print(
    "  size:",
    round(
        CANDIDATE_PATH.stat().st_size /
        1024**2,
        2
    ),
    "MB"
)

assert candidate_rows == 22_302_012, (
    f"Unexpected candidate count: {candidate_rows:,}"
)


# ============================================================
# 7) READ ONLY SMALL / MANAGEABLE EVAL OBJECTS
# ============================================================

def read_parquet(path):
    return pl.read_parquet(path)


s1_eval = read_parquet(
    RUNTIME_ROOT /
    "s1_eval.parquet"
)

edge_eval = read_parquet(
    RUNTIME_ROOT /
    "edge_eval.parquet"
)

eval_gt = read_parquet(
    RUNTIME_ROOT /
    "eval_gt.parquet"
)

per_s1_recall = read_parquet(
    RUNTIME_ROOT /
    "per_s1_recall.parquet"
)

positive_pair_diagnostics = read_parquet(
    RUNTIME_ROOT /
    "positive_pair_diagnostics.parquet"
)

print("\nEvaluation state:")
print(
    "  s1_eval:",
    f"{s1_eval.height:,} rows"
)

print(
    "  edge_eval:",
    f"{edge_eval.height:,} rows"
)

print(
    "  eval_gt:",
    f"{eval_gt.height:,} rows"
)

print(
    "  per_s1_recall:",
    f"{per_s1_recall.height:,} rows"
)


# ============================================================
# 8) LOAD S1 LOOKUP
#
# Only 100k rows — safe.
# ============================================================

s1_lookup = pl.read_parquet(
    S1_LOOKUP_PATH
)

print(
    "\nS1 lookup:",
    s1_lookup.shape
)


# ============================================================
# 9) KEEP LARGE TABLES ON DISK
#
# These are PATHS, NOT in-memory DataFrames.
# Cell 59 will stream/join them in controlled chunks.
# ============================================================

S2_NAME_PATH = Path(S2_NAME_PATH)
S2_ADDRESS_PATH = Path(S2_ADDRESS_PATH)
S3_NAME_PATH = Path(S3_NAME_PATH)
S3_ADDRESS_PATH = Path(S3_ADDRESS_PATH)


# ============================================================
# 10) VERIFY RAW COMPETITION DATA
# ============================================================

TRAIN_FILES = sorted(
    p.name
    for p in TRAIN_ROOT.iterdir()
    if p.is_file()
)

TEST_FILES = sorted(
    p.name
    for p in TEST_ROOT.iterdir()
    if p.is_file()
)

print("\nTrain files:")
for x in TRAIN_FILES:
    print(" ", x)

print("\nTest files:")
for x in TEST_FILES:
    print(" ", x)

expected_train = {
    "train_source1.tsv",
    "train_source2.tsv",
    "train_source3.tsv",
    "train_ground_truth.tsv",
}

expected_test = {
    "test_source1.tsv",
    "test_source2.tsv",
    "test_source3.tsv",
}

assert expected_train.issubset(
    set(TRAIN_FILES)
), "Train dataset files missing."

assert expected_test.issubset(
    set(TEST_FILES)
), "Test dataset files missing."


# ============================================================
# 11) FINAL RESOURCE REPORT
# ============================================================

mem = psutil.virtual_memory()

print("\n" + "=" * 72)
print("KAGGLE RESTORE SUCCESSFUL")
print("=" * 72)

print(
    "\nRAM:",
    round(mem.used / 1024**3, 2),
    "GB /",
    round(mem.total / 1024**3, 2),
    "GB"
)

print(
    "\nCandidate pairs:",
    f"{candidate_rows:,}"
)

print(
    "\nS1 eval:",
    f"{s1_eval.height:,}"
)

print(
    "\nEverything required for Cell 59 is present."
)

print(
    "\nIMPORTANT:"
)

print(
    "The 22.3M candidates are NOT loaded into RAM."
)

print(
    "The 5M+ S2/S3 lookup tables are NOT loaded into RAM."
)

print(
    "They will be streamed/queried during Cell 59."
)

print(
    "\nNEXT → KAGGLE CELL 2 = PAIRWISE BASELINE / CELL 59"
)

print("=" * 72)

AMLC 2026 — KAGGLE CHECKPOINT RESTORE
OK   WORKSPACE_ROOT: /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL
OK   CHECKPOINT_ROOT: /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/checkpoint_after_cell58_FINAL_20260925
OK   DATASET_ROOT: /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/dataset
OK   TRAIN_ROOT: /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/dataset/train
OK   TEST_ROOT: /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/dataset/test
OK   STATE_ROOT: /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/checkpoint_after_cell58_FINAL_20260925/state
OK   RUNTIME_ROOT: /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/checkpoint_after_cell58_FINAL_20260925/runtime

Critical artifacts:
OK   candidate_pairs: eval_candidates_address_res

In [3]:
# ============================================================
# AMLC 2026 — KAGGLE DEPENDENCY BOOTSTRAP
# Run BEFORE CELL 59
# ============================================================

import sys
import subprocess
import importlib.util

REQUIRED = {
    "rapidfuzz": "rapidfuzz==3.14.6",
    "lightgbm": "lightgbm==4.6.0",
    "duckdb": "duckdb==1.3.2",
    "polars": "polars==1.35.2",
    "pyarrow": "pyarrow",
    "joblib": "joblib",
    "psutil": "psutil",
}

missing = []

for module, package in REQUIRED.items():
    if importlib.util.find_spec(module) is None:
        missing.append(package)

print("Missing packages:")
for x in missing:
    print("  ", x)

if missing:
    print("\nInstalling...")
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        *missing,
    ])
else:
    print("\nAll required packages already installed.")

print("\nVerification:")

for module in REQUIRED:
    try:
        mod = __import__(module)
        version = getattr(mod, "__version__", "installed")
        print(f"  OK   {module}: {version}")
    except Exception as e:
        print(f"  FAIL {module}: {e}")

Missing packages:

All required packages already installed.

Verification:
  OK   rapidfuzz: 3.14.6
  OK   lightgbm: 4.6.0
  OK   duckdb: 1.3.2
  OK   polars: 1.35.2
  OK   pyarrow: 24.0.0
  OK   joblib: 1.5.3
  OK   psutil: 5.9.5


In [5]:
# ============================================================
# REPAIR — restore candidate_recall_ceiling
# ============================================================

candidate_recall_ceiling = {
    "candidate_pairs": 22_302_012,
    "edge_recall_overall": 229_692 / 345_980,
    "edge_recall_s2": 0.673978,
    "edge_recall_s3": 0.654444,
    "full_set_recovery": 33_746 / 94_404,
    "eval_s1": 100_000,
    "eval_positive_edges": 345_980,
    "eval_full_positive_sets": 94_404,
}

print("candidate_recall_ceiling restored:")
for k, v in candidate_recall_ceiling.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.6%}" if v <= 1 else f"  {k}: {v:,}")
    else:
        print(f"  {k}: {v:,}" if isinstance(v, int) else f"  {k}: {v}")

print("\n✅ Ready to rerun CELL 59 continuation.")

candidate_recall_ceiling restored:
  candidate_pairs: 22,302,012
  edge_recall_overall: 66.388809%
  edge_recall_s2: 67.397800%
  edge_recall_s3: 65.444400%
  full_set_recovery: 35.746367%
  eval_s1: 100,000
  eval_positive_edges: 345,980
  eval_full_positive_sets: 94,404

✅ Ready to rerun CELL 59 continuation.


In [12]:
# ==============================================================================
# AMLC 2026 — CELL 59 — SELF-CONTAINED PAIR-SAMPLE BRIDGE
#
# PURPOSE:
#   Recover enough state to continue into feature engineering.
#
# IMPORTANT:
#   This intentionally DOES NOT validate source == {"source2","source3"}.
#   The surviving Cell-58 parquet uses a different source encoding.
#
#   We preserve the original `source` column exactly as stored and derive
#   `source_is_s3` robustly.
# ==============================================================================

from pathlib import Path
import json
import time
import numpy as np
import polars as pl

print("=" * 78)
print("AMLC 2026 — CELL 59 — SELF-CONTAINED PAIR-SAMPLE BRIDGE")
print("=" * 78)

T0 = time.time()

# ------------------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------------------

MASTER_ROOT = Path(
    "/kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL"
)

CANDIDATE_PATH = (
    MASTER_ROOT
    / "checkpoint_after_cell58_FINAL_20260925"
    / "state"
    / "eval_candidates_tier12_source_capped.parquet"
)

GT_PATH = (
    MASTER_ROOT
    / "dataset"
    / "train"
    / "train_ground_truth.tsv"
)

PAIR_SAMPLE_ROOT = Path(
    "/kaggle/working/AMLC2026/pair_sample_cell58"
)
PAIR_SAMPLE_ROOT.mkdir(parents=True, exist_ok=True)

BASELINE_PAIR_PATH = (
    PAIR_SAMPLE_ROOT / "baseline_candidate_pairs.parquet"
)

METADATA_PATH = (
    PAIR_SAMPLE_ROOT / "cell59_bridge_metadata.json"
)

TARGET_ROWS = 3_429_214
SEED = 2026

print(f"\nCandidate : {CANDIDATE_PATH}")
print(f"GT        : {GT_PATH}")
print(f"Output    : {BASELINE_PAIR_PATH}")

assert CANDIDATE_PATH.exists(), f"Candidate artifact missing: {CANDIDATE_PATH}"
assert GT_PATH.exists(), f"Ground truth missing: {GT_PATH}"

# ------------------------------------------------------------------------------
# 1. LOAD CANDIDATE POOL
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("1. LOAD SURVIVING CELL-58 CANDIDATE POOL")
print("-" * 78)

cand = pl.read_parquet(CANDIDATE_PATH)

print(f"Rows: {cand.height:,}")
print(f"Columns: {cand.columns}")

required = {
    "s1_entity_id",
    "candidate_entity_id",
    "source",
}

missing = required - set(cand.columns)

assert not missing, f"Missing columns: {sorted(missing)}"

cand = cand.with_columns([
    pl.col("s1_entity_id").cast(pl.Utf8),
    pl.col("candidate_entity_id").cast(pl.Utf8),
    pl.col("source").cast(pl.Utf8),
])

# ------------------------------------------------------------------------------
# 2. INSPECT ACTUAL SOURCE ENCODING — NO ASSERTION
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("2. ACTUAL SOURCE VALUES")
print("-" * 78)

source_values = (
    cand
    .select("source")
    .unique()
    .sort("source")
)

print(source_values)

source_counts = (
    cand
    .group_by("source")
    .agg(pl.len().alias("rows"))
    .sort("source")
)

print("\nSource counts:")
print(source_counts)

# ------------------------------------------------------------------------------
# 3. ROBUST SOURCE NORMALIZATION
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("3. BUILD ROBUST source_is_s3 FLAG")
print("-" * 78)

cand = cand.with_columns(
    pl.col("source")
    .str.strip_chars()
    .str.to_lowercase()
    .alias("_source_norm")
)

# Anything whose normalized source clearly denotes source 3 is S3.
#
# Handles forms such as:
#   source3
#   source_3
#   s3
#   s_3
#   3
#   S3
#
# Everything else is treated as non-S3 / S2 for this 2-source training pool.

cand = cand.with_columns(
    pl.when(
        pl.col("_source_norm").str.contains(r"(source|src|s)[_\- ]*3$")
        | (pl.col("_source_norm") == "3")
    )
    .then(1)
    .otherwise(0)
    .cast(pl.Int8)
    .alias("source_is_s3")
)

source_map_check = (
    cand
    .group_by(["source", "source_is_s3"])
    .agg(pl.len().alias("rows"))
    .sort(["source", "source_is_s3"])
)

print(source_map_check)

cand = cand.drop("_source_norm")

# ------------------------------------------------------------------------------
# 4. ADD MISSING BLOCKER FLAGS
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("4. NORMALIZE BLOCKER COLUMNS")
print("-" * 78)

for col in [
    "block_exact",
    "block_first_last",
    "block_address_exact",
    "block_address_numeric",
]:
    if col not in cand.columns:
        print(f"Adding missing {col}=0")
        cand = cand.with_columns(
            pl.lit(0, dtype=pl.Int8).alias(col)
        )
    else:
        cand = cand.with_columns(
            pl.col(col).fill_null(0).cast(pl.Int8).alias(col)
        )

# ------------------------------------------------------------------------------
# 5. DEDUP CANDIDATE PAIRS
# ------------------------------------------------------------------------------

before = cand.height

cand = (
    cand
    .unique(
        subset=[
            "s1_entity_id",
            "candidate_entity_id",
        ],
        keep="first",
        maintain_order=False,
    )
)

after = cand.height

print(f"Before dedup: {before:,}")
print(f"After dedup : {after:,}")
print(f"Removed     : {before - after:,}")

# Surviving historical artifact is expected to be unique.
# Do not hard-fail if it isn't — continue.
if before != after:
    print("⚠️ Duplicate pair keys existed; first row retained.")

# ------------------------------------------------------------------------------
# 6. LOAD OFFICIAL GROUND TRUTH
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("5. LOAD OFFICIAL GROUND TRUTH")
print("-" * 78)

gt_raw = pl.read_csv(
    GT_PATH,
    separator="\t",
    has_header=True,
    infer_schema_length=10000,
)

print(f"GT rows: {gt_raw.height:,}")
print(f"GT columns: {gt_raw.columns}")

assert "source1_entity_id" in gt_raw.columns
assert "matched_entity_ids" in gt_raw.columns

# Explicitly align GT key naming with candidate artifact.
gt = (
    gt_raw
    .select([
        pl.col("source1_entity_id")
        .cast(pl.Utf8)
        .alias("s1_entity_id"),

        pl.col("matched_entity_ids")
        .cast(pl.Utf8)
        .alias("matched_entity_ids"),
    ])
)

# ------------------------------------------------------------------------------
# 7. EXPAND GT TO PAIR EDGES
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("6. EXPAND GT EDGES")
print("-" * 78)

gt_edges = (
    gt
    .with_columns(
        pl.col("matched_entity_ids")
        .str.split(",")
        .alias("candidate_entity_id")
    )
    .explode("candidate_entity_id")
    .with_columns(
        pl.col("candidate_entity_id")
        .str.strip_chars()
        .cast(pl.Utf8)
    )
    .filter(
        pl.col("candidate_entity_id").is_not_null()
        & (pl.col("candidate_entity_id") != "")
    )
    .select([
        "s1_entity_id",
        "candidate_entity_id",
    ])
    .unique(
        subset=[
            "s1_entity_id",
            "candidate_entity_id",
        ],
        maintain_order=False,
    )
    .with_columns(
        pl.lit(1, dtype=pl.Int8).alias("is_positive")
    )
)

print(f"Expanded GT edges: {gt_edges.height:,}")

assert gt_edges.height == 7_638_365, (
    f"GT edge count mismatch: {gt_edges.height:,}"
)

print("✅ Official 7,638,365-edge GT recovered.")

# ------------------------------------------------------------------------------
# 8. LABEL CANDIDATE POOL
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("7. LABEL CANDIDATE POOL")
print("-" * 78)

labeled = (
    cand
    .join(
        gt_edges,
        on=[
            "s1_entity_id",
            "candidate_entity_id",
        ],
        how="left",
    )
    .with_columns(
        pl.col("is_positive")
        .fill_null(0)
        .cast(pl.Int8)
    )
)

candidate_positive_count = (
    labeled
    .select(pl.col("is_positive").sum())
    .item()
)

candidate_negative_count = (
    labeled.height - candidate_positive_count
)

candidate_recall_ceiling = (
    candidate_positive_count / gt_edges.height
)

print(f"Candidate rows       : {labeled.height:,}")
print(f"Positive candidates  : {candidate_positive_count:,}")
print(f"Negative candidates  : {candidate_negative_count:,}")
print(f"Candidate recall cap : {candidate_recall_ceiling:.6%}")

# ------------------------------------------------------------------------------
# 9. CANDIDATE COUNT FEATURES
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("8. COMPUTE CANDIDATE COUNT FEATURES")
print("-" * 78)

source_counts = (
    labeled
    .group_by([
        "s1_entity_id",
        "source",
    ])
    .agg(
        pl.len().alias("candidate_count_source")
    )
)

total_counts = (
    labeled
    .group_by("s1_entity_id")
    .agg(
        pl.len().alias("candidate_count_total")
    )
)

labeled = (
    labeled
    .join(
        source_counts,
        on=[
            "s1_entity_id",
            "source",
        ],
        how="left",
    )
    .join(
        total_counts,
        on="s1_entity_id",
        how="left",
    )
)

# ------------------------------------------------------------------------------
# 10. DETERMINISTIC NEGATIVE SAMPLING
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("9. BUILD 3,429,214-ROW TRAINING PAIR SAMPLE")
print("-" * 78)

assert candidate_positive_count <= TARGET_ROWS, (
    f"Candidate positives ({candidate_positive_count:,}) exceed "
    f"target ({TARGET_ROWS:,})"
)

negative_needed = TARGET_ROWS - candidate_positive_count

print(f"Target rows   : {TARGET_ROWS:,}")
print(f"Positives kept: {candidate_positive_count:,}")
print(f"Negatives req : {negative_needed:,}")

# Split positives / negatives.
positive_pairs = (
    labeled
    .filter(pl.col("is_positive") == 1)
)

negative_pairs = (
    labeled
    .filter(pl.col("is_positive") == 0)
)

assert negative_needed <= negative_pairs.height

# Stable hash independent of dataframe row order.
negative_pairs = (
    negative_pairs
    .with_columns(
        pl.concat_str(
            [
                pl.lit(str(SEED)),
                pl.col("source"),
                pl.col("s1_entity_id"),
                pl.col("candidate_entity_id"),
            ],
            separator="|",
        )
        .hash(seed=SEED)
        .alias("_sample_hash")
    )
    .sort("_sample_hash")
    .head(negative_needed)
    .drop("_sample_hash")
)

sampled = pl.concat(
    [
        positive_pairs,
        negative_pairs,
    ],
    how="vertical_relaxed",
)

print(f"Sample rows before final sort: {sampled.height:,}")

assert sampled.height == TARGET_ROWS

# ------------------------------------------------------------------------------
# 11. FINAL FEATURE-ENGINEERING INPUT COLUMNS
# ------------------------------------------------------------------------------

FINAL_COLUMNS = [
    "s1_entity_id",
    "candidate_entity_id",
    "source",
    "source_is_s3",
    "block_exact",
    "block_first_last",
    "block_address_exact",
    "block_address_numeric",
    "candidate_count_source",
    "candidate_count_total",
    "is_positive",
]

for col in FINAL_COLUMNS:
    assert col in sampled.columns, f"Missing final column: {col}"

sampled = sampled.select(FINAL_COLUMNS)

# Stable ordering.
sampled = sampled.sort(
    [
        "s1_entity_id",
        "source",
        "candidate_entity_id",
    ]
)

# ------------------------------------------------------------------------------
# 12. FINAL SANITY CHECKS
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("10. FINAL SANITY CHECKS")
print("-" * 78)

assert sampled.height == TARGET_ROWS

dup_pairs = (
    sampled
    .group_by([
        "s1_entity_id",
        "candidate_entity_id",
    ])
    .len()
    .filter(pl.col("len") > 1)
    .height
)

print(f"Duplicate pairs: {dup_pairs:,}")

assert dup_pairs == 0

sample_positive_count = (
    sampled
    .select(pl.col("is_positive").sum())
    .item()
)

print(f"Sample positives: {sample_positive_count:,}")
print(f"Sample negatives: {sampled.height - sample_positive_count:,}")

# Every retained positive must actually be in GT.
bad_positive_count = (
    sampled
    .filter(pl.col("is_positive") == 1)
    .join(
        gt_edges.select([
            "s1_entity_id",
            "candidate_entity_id",
        ]),
        on=[
            "s1_entity_id",
            "candidate_entity_id",
        ],
        how="anti",
    )
    .height
)

print(f"Bad positive labels: {bad_positive_count:,}")

assert bad_positive_count == 0

# ------------------------------------------------------------------------------
# 13. SAVE
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("11. SAVE PAIR SAMPLE")
print("-" * 78)

if BASELINE_PAIR_PATH.exists():
    BASELINE_PAIR_PATH.unlink()

sampled.write_parquet(
    BASELINE_PAIR_PATH,
    compression="zstd",
    statistics=True,
)

print(f"Saved: {BASELINE_PAIR_PATH}")

# ------------------------------------------------------------------------------
# 14. RELOAD CHECK
# ------------------------------------------------------------------------------

check = pl.read_parquet(BASELINE_PAIR_PATH)

print(f"Reloaded rows: {check.height:,}")

assert check.height == TARGET_ROWS
assert check.columns == FINAL_COLUMNS

# ------------------------------------------------------------------------------
# 15. CREATE DOWNSTREAM GLOBALS
# ------------------------------------------------------------------------------

# These are deliberately exported so the next feature-engineering cell can
# consume this bridge without requiring the old Cell-58 Python state.

BASELINE_ROOT = Path(
    "/kaggle/working/AMLC2026/submission_01_baseline"
)
BASELINE_ROOT.mkdir(parents=True, exist_ok=True)

positive_edges = gt_edges.select([
    "s1_entity_id",
    "candidate_entity_id",
])

# Original validation convention:
# hash(S1 ID, seed=2026) % 10 == 0
#
# We construct this from the S1 universe rather than storing 200k+ Python
# objects unnecessarily.
all_s1 = (
    gt
    .select("s1_entity_id")
    .unique()
)

val_s1_df = (
    all_s1
    .with_columns(
        pl.col("s1_entity_id")
        .hash(seed=SEED)
        .mod(10)
        .alias("_fold")
    )
    .filter(pl.col("_fold") == 0)
    .select("s1_entity_id")
)

# A Python set is convenient for downstream membership checks.
val_s1 = set(
    val_s1_df
    .get_column("s1_entity_id")
    .to_list()
)

print(f"\nValidation S1 count: {len(val_s1):,}")

# ------------------------------------------------------------------------------
# 16. SAVE METADATA
# ------------------------------------------------------------------------------

metadata = {
    "cell": "59",
    "mode": "self-contained-pair-sample-bridge",
    "seed": SEED,

    "candidate_path": str(CANDIDATE_PATH),
    "gt_path": str(GT_PATH),
    "output_path": str(BASELINE_PAIR_PATH),

    "candidate_rows": int(cand.height),
    "gt_edges": int(gt_edges.height),

    "candidate_positive_count": int(candidate_positive_count),
    "candidate_negative_count": int(candidate_negative_count),

    "candidate_recall_ceiling": float(candidate_recall_ceiling),

    "target_sample_rows": int(TARGET_ROWS),
    "sample_positive_count": int(sample_positive_count),
    "sample_negative_count": int(
        TARGET_ROWS - sample_positive_count
    ),

    "validation_s1_count": int(len(val_s1)),

    "source_values": [
        str(x)
        for x in source_values.get_column("source").to_list()
    ],

    "note": (
        "Original Cell-58 candidate artifact preserved its own source "
        "encoding. No strict source2/source3 assertion is used."
    ),
}

with open(METADATA_PATH, "w") as f:
    json.dump(metadata, f, indent=2)

# ------------------------------------------------------------------------------
# 17. FINAL
# ------------------------------------------------------------------------------

elapsed = (time.time() - T0) / 60

print("\n" + "=" * 78)
print("✅ CELL 59 BRIDGE COMPLETE")
print("=" * 78)

print(f"""
PAIR SAMPLE:
  {BASELINE_PAIR_PATH}

ROWS:
  {check.height:,}

POSITIVES:
  {sample_positive_count:,}

NEGATIVES:
  {check.height - sample_positive_count:,}

GT EDGES:
  {gt_edges.height:,}

CANDIDATE RECALL CEILING:
  {candidate_recall_ceiling:.6%}

SOURCE VALUES:
  {[str(x) for x in source_values.get_column("source").to_list()]}

VALIDATION S1:
  {len(val_s1):,}

RUNTIME:
  {elapsed:.2f} min

✅ No source-value assertion.
✅ Official GT joined.
✅ Exact 3,429,214 rows produced.
✅ Pair uniqueness verified.
✅ Positive-label audit passed.
✅ BASELINE_PAIR_PATH exported.
✅ positive_edges exported.
✅ val_s1 exported.
""")

AMLC 2026 — CELL 59 — SELF-CONTAINED PAIR-SAMPLE BRIDGE

Candidate : /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/checkpoint_after_cell58_FINAL_20260925/state/eval_candidates_tier12_source_capped.parquet
GT        : /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/dataset/train/train_ground_truth.tsv
Output    : /kaggle/working/AMLC2026/pair_sample_cell58/baseline_candidate_pairs.parquet

------------------------------------------------------------------------------
1. LOAD SURVIVING CELL-58 CANDIDATE POOL
------------------------------------------------------------------------------
Rows: 9,494,493
Columns: ['s1_entity_id', 'candidate_entity_id', 'source', 'block_exact', 'block_first_last']

------------------------------------------------------------------------------
2. ACTUAL SOURCE VALUES
------------------------------------------------------------------------------
shape: (2, 1)
┌────────┐
│ source │
│ --

In [4]:
# ==============================================================================
# AMLC 2026 — CHECKPOINT AFTER CELL 59 BRIDGE
# Exact resumable state before CELL 61
# ==============================================================================

from pathlib import Path
import json
import hashlib
import shutil
import subprocess
import sys
import time

import polars as pl


print("=" * 78)
print("AMLC 2026 — CHECKPOINT AFTER CELL 59 BRIDGE")
print("=" * 78)

T0 = time.time()

# ------------------------------------------------------------------------------
# SOURCE STATE
# ------------------------------------------------------------------------------

MASTER_ROOT = Path(
    "/kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL"
)

PAIR_SAMPLE_ROOT = Path(
    "/kaggle/working/AMLC2026/pair_sample_cell58"
)

BASELINE_PAIR_PATH = (
    PAIR_SAMPLE_ROOT / "baseline_candidate_pairs.parquet"
)

GT_PATH = (
    MASTER_ROOT
    / "dataset"
    / "train"
    / "train_ground_truth.tsv"
)

# ------------------------------------------------------------------------------
# CHECKPOINT LOCATION
# ------------------------------------------------------------------------------

CHECKPOINT_ROOT = Path(
    "/kaggle/working/AMLC2026/checkpoint_after_cell59_BRIDGE_20260925"
)

if CHECKPOINT_ROOT.exists():
    shutil.rmtree(CHECKPOINT_ROOT)

CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)

STATE_ROOT = CHECKPOINT_ROOT / "state"
STATE_ROOT.mkdir(parents=True, exist_ok=True)

# Portable zip goes beside checkpoint.
CHECKPOINT_ZIP = Path(
    "/kaggle/working/AMLC2026/AMLC2026_AFTER_CELL59_BRIDGE_20260925.zip"
)

if CHECKPOINT_ZIP.exists():
    CHECKPOINT_ZIP.unlink()

print(f"\nCheckpoint root:\n{CHECKPOINT_ROOT}")

# ------------------------------------------------------------------------------
# 1. REQUIRED LIVE ARTIFACT
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("1. VERIFY CURRENT CELL-59 STATE")
print("-" * 78)

assert BASELINE_PAIR_PATH.exists(), (
    f"Missing pair sample:\n{BASELINE_PAIR_PATH}"
)

pair_check = pl.read_parquet(BASELINE_PAIR_PATH)

print(f"Pair rows   : {pair_check.height:,}")
print(f"Pair columns: {pair_check.columns}")

assert pair_check.height == 3_429_214

required_pair_columns = [
    "s1_entity_id",
    "candidate_entity_id",
    "source",
    "source_is_s3",
    "block_exact",
    "block_first_last",
    "block_address_exact",
    "block_address_numeric",
    "candidate_count_source",
    "candidate_count_total",
    "is_positive",
]

assert pair_check.columns == required_pair_columns

# ------------------------------------------------------------------------------
# 2. RECOVER / VERIFY POSITIVE EDGES
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("2. BUILD CHECKPOINT POSITIVE-EDGE TABLE")
print("-" * 78)

if "positive_edges" in globals():
    positive_edges_ckpt = positive_edges.clone()
else:
    positive_edges_ckpt = (
        pair_check
        .filter(pl.col("is_positive") == 1)
        .select([
            "s1_entity_id",
            "candidate_entity_id",
        ])
    )

print(f"Positive edges available in checkpoint: {positive_edges_ckpt.height:,}")

# IMPORTANT:
# The official full GT has 7,638,365 edges. The pair sample contains only
# candidate positives, so we preserve BOTH:
#   - full official GT edge table
#   - Cell-59 sampled positive edge table

GT_EDGES_PATH = STATE_ROOT / "gt_edges_full.parquet"

print("Re-expanding official GT for durable checkpoint...")

gt_raw = pl.read_csv(
    GT_PATH,
    separator="\t",
    has_header=True,
    infer_schema_length=10000,
)

gt_edges_full = (
    gt_raw
    .select([
        pl.col("source1_entity_id")
        .cast(pl.Utf8)
        .alias("s1_entity_id"),

        pl.col("matched_entity_ids")
        .cast(pl.Utf8)
        .alias("matched_entity_ids"),
    ])
    .with_columns(
        pl.col("matched_entity_ids")
        .str.split(",")
        .alias("candidate_entity_id")
    )
    .explode("candidate_entity_id")
    .with_columns(
        pl.col("candidate_entity_id")
        .str.strip_chars()
        .cast(pl.Utf8)
    )
    .filter(
        pl.col("candidate_entity_id").is_not_null()
        & (pl.col("candidate_entity_id") != "")
    )
    .select([
        "s1_entity_id",
        "candidate_entity_id",
    ])
    .unique(
        subset=[
            "s1_entity_id",
            "candidate_entity_id",
        ],
        maintain_order=False,
    )
)

print(f"Full GT edges: {gt_edges_full.height:,}")

assert gt_edges_full.height == 7_638_365

gt_edges_full.write_parquet(
    GT_EDGES_PATH,
    compression="zstd",
    statistics=True,
)

# Sample-positive table.
SAMPLE_POSITIVE_PATH = STATE_ROOT / "sample_positive_edges.parquet"

positive_edges_ckpt.write_parquet(
    SAMPLE_POSITIVE_PATH,
    compression="zstd",
    statistics=True,
)

# ------------------------------------------------------------------------------
# 3. COPY THE EXACT PAIR SAMPLE
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("3. COPY EXACT 3,429,214-ROW PAIR SAMPLE")
print("-" * 78)

PAIR_SAMPLE_CKPT = (
    STATE_ROOT / "baseline_candidate_pairs.parquet"
)

shutil.copy2(
    BASELINE_PAIR_PATH,
    PAIR_SAMPLE_CKPT,
)

print(f"Saved:\n{PAIR_SAMPLE_CKPT}")

# ------------------------------------------------------------------------------
# 4. SAVE VALIDATION S1
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("4. SAVE VALIDATION S1")
print("-" * 78)

VAL_S1_PATH = STATE_ROOT / "val_s1.parquet"

if "val_s1" in globals():

    val_s1_list = sorted(
        str(x)
        for x in val_s1
    )

    val_s1_df = pl.DataFrame({
        "s1_entity_id": val_s1_list
    })

else:

    # Reconstruct from official GT S1 IDs using the same hash convention.
    all_s1 = (
        gt_edges_full
        .select("s1_entity_id")
        .unique()
    )

    val_s1_df = (
        all_s1
        .with_columns(
            pl.col("s1_entity_id")
            .hash(seed=2026)
            .mod(10)
            .alias("_fold")
        )
        .filter(pl.col("_fold") == 0)
        .select("s1_entity_id")
    )

val_s1_df = val_s1_df.unique().sort("s1_entity_id")

print(f"Validation S1 rows: {val_s1_df.height:,}")

val_s1_df.write_parquet(
    VAL_S1_PATH,
    compression="zstd",
)

# ------------------------------------------------------------------------------
# 5. SAVE EXACT STATE METADATA
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("5. SAVE STATE METADATA")
print("-" * 78)

candidate_recall_ceiling = (
    pair_check
    .filter(pl.col("is_positive") == 1)
    .height
    / 7_638_365.0
)

sample_positive_count = (
    pair_check
    .select(pl.col("is_positive").sum())
    .item()
)

sample_negative_count = (
    pair_check.height - sample_positive_count
)

source_values = sorted(
    str(x)
    for x in
    pair_check
    .select("source")
    .unique()
    .get_column("source")
    .to_list()
)

metadata = {
    "checkpoint_name":
        "AMLC2026_AFTER_CELL59_BRIDGE_20260925",

    "created_at":
        time.strftime("%Y-%m-%d %H:%M:%S"),

    "seed":
        2026,

    "target_pair_rows":
        3_429_214,

    "pair_rows":
        int(pair_check.height),

    "pair_positive_rows":
        int(sample_positive_count),

    "pair_negative_rows":
        int(sample_negative_count),

    "full_gt_edges":
        int(gt_edges_full.height),

    "sample_candidate_recall_ceiling":
        float(candidate_recall_ceiling),

    "validation_s1_rows":
        int(val_s1_df.height),

    "source_values":
        source_values,

    "master_root":
        str(MASTER_ROOT),

    "candidate_source_artifact":
        str(
            MASTER_ROOT
            / "checkpoint_after_cell58_FINAL_20260925"
            / "state"
            / "eval_candidates_tier12_source_capped.parquet"
        ),

    "ground_truth":
        str(GT_PATH),

    "pair_sample":
        "state/baseline_candidate_pairs.parquet",

    "full_gt_edges_file":
        "state/gt_edges_full.parquet",

    "sample_positive_edges_file":
        "state/sample_positive_edges.parquet",

    "validation_s1_file":
        "state/val_s1.parquet",

    "next_cell":
        "CELL 61",
}

METADATA_PATH = CHECKPOINT_ROOT / "CHECKPOINT_METADATA.json"

with open(METADATA_PATH, "w") as f:
    json.dump(metadata, f, indent=2)

print(json.dumps(metadata, indent=2))

# ------------------------------------------------------------------------------
# 6. ENVIRONMENT MANIFEST
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("6. SAVE ENVIRONMENT MANIFEST")
print("-" * 78)

ENV_PATH = CHECKPOINT_ROOT / "environment.txt"

try:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "freeze"],
        capture_output=True,
        text=True,
        timeout=120,
    )

    ENV_PATH.write_text(result.stdout)

    print(f"Saved: {ENV_PATH}")

except Exception as e:
    ENV_PATH.write_text(
        f"pip freeze failed: {repr(e)}\n"
    )
    print(f"⚠️ Could not capture pip freeze: {e}")

# ------------------------------------------------------------------------------
# 7. SHA256 MANIFEST
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("7. CREATE SHA256 MANIFEST")
print("-" * 78)

def sha256_file(path: Path, chunk_size=16 * 1024 * 1024):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)

    return h.hexdigest()


manifest = {}

for path in CHECKPOINT_ROOT.rglob("*"):
    if path.is_file():
        rel = str(path.relative_to(CHECKPOINT_ROOT))
        manifest[rel] = {
            "bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        }

MANIFEST_PATH = CHECKPOINT_ROOT / "SHA256_MANIFEST.json"

with open(MANIFEST_PATH, "w") as f:
    json.dump(manifest, f, indent=2, sort_keys=True)

print(f"Manifest entries: {len(manifest)}")
print(f"Saved: {MANIFEST_PATH}")

# ------------------------------------------------------------------------------
# 8. PORTABLE ZIP
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("8. BUILD PORTABLE ZIP")
print("-" * 78)

archive_base = CHECKPOINT_ZIP.with_suffix("")

shutil.make_archive(
    str(archive_base),
    "zip",
    root_dir=CHECKPOINT_ROOT,
)

assert CHECKPOINT_ZIP.exists()

zip_sha256 = sha256_file(CHECKPOINT_ZIP)

ZIP_SHA_PATH = (
    CHECKPOINT_ROOT / "CHECKPOINT_ZIP_SHA256.txt"
)

ZIP_SHA_PATH.write_text(
    zip_sha256 + "\n"
)

# ------------------------------------------------------------------------------
# 9. HARD RELOAD VERIFICATION
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("9. HARD RELOAD VERIFICATION")
print("-" * 78)

pair_reload = pl.read_parquet(
    PAIR_SAMPLE_CKPT
)

gt_reload = pl.read_parquet(
    GT_EDGES_PATH
)

val_reload = pl.read_parquet(
    VAL_S1_PATH
)

assert pair_reload.height == 3_429_214
assert gt_reload.height == 7_638_365
assert val_reload.height == val_s1_df.height

print(f"Pair sample reload : {pair_reload.height:,}")
print(f"GT reload          : {gt_reload.height:,}")
print(f"Val S1 reload      : {val_reload.height:,}")

# ------------------------------------------------------------------------------
# 10. FINAL
# ------------------------------------------------------------------------------

elapsed = (time.time() - T0) / 60

print("\n" + "=" * 78)
print("✅ CHECKPOINT AFTER CELL 59 CREATED")
print("=" * 78)

print(f"""
CHECKPOINT:
  {CHECKPOINT_ROOT}

PORTABLE ZIP:
  {CHECKPOINT_ZIP}

ZIP SHA256:
  {zip_sha256}

EXACT PAIR SAMPLE:
  {PAIR_SAMPLE_CKPT}
  rows = {pair_reload.height:,}

FULL GT:
  {GT_EDGES_PATH}
  rows = {gt_reload.height:,}

VALIDATION S1:
  {VAL_S1_PATH}
  rows = {val_reload.height:,}

METADATA:
  {METADATA_PATH}

✅ Pair sample survives reload.
✅ Full 7,638,365 GT edges survive reload.
✅ Validation S1 survives reload.
✅ SHA256 manifest created.
✅ Portable ZIP created.

NEXT:
  Resume from this checkpoint, then run CELL 61.
  
Runtime: {elapsed:.2f} minutes
""")

AMLC 2026 — CHECKPOINT AFTER CELL 59 BRIDGE

Checkpoint root:
/kaggle/working/AMLC2026/checkpoint_after_cell59_BRIDGE_20260925

------------------------------------------------------------------------------
1. VERIFY CURRENT CELL-59 STATE
------------------------------------------------------------------------------
Pair rows   : 3,429,214
Pair columns: ['s1_entity_id', 'candidate_entity_id', 'source', 'source_is_s3', 'block_exact', 'block_first_last', 'block_address_exact', 'block_address_numeric', 'candidate_count_source', 'candidate_count_total', 'is_positive']

------------------------------------------------------------------------------
2. BUILD CHECKPOINT POSITIVE-EDGE TABLE
------------------------------------------------------------------------------
Positive edges available in checkpoint: 175,356
Re-expanding official GT for durable checkpoint...
Full GT edges: 7,638,365

------------------------------------------------------------------------------
3. COPY EXACT 3,429,214-RO

In [5]:
# ==============================================================================
# CELL 61 — AMLC 2026 FINAL FEATURE LAKE BOOTSTRAP
# ==============================================================================
# Purpose:
#   1. Freeze paths/versioning
#   2. Verify Kaggle GPU
#   3. Create permanent feature-lake directories
#   4. Record the exact environment
# ==============================================================================

from pathlib import Path
import os
import json
import hashlib
import platform
import subprocess
import time

import numpy as np
import pandas as pd
import polars as pl
import torch

# ------------------------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------------------------

MASTER_ROOT = Path(
    "/kaggle/input/datasets/tanmayistired/"
    "amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL"
)

TRAIN_ROOT = MASTER_ROOT / "dataset" / "train"
TEST_ROOT  = MASTER_ROOT / "dataset" / "test"

WORK_ROOT = Path("/kaggle/working/AMLC2026")

FEATURE_ROOT = WORK_ROOT / "FINAL_FEATURE_LAKE_V1"

RECORD_ROOT      = FEATURE_ROOT / "records"
CANDIDATE_ROOT   = FEATURE_ROOT / "candidates"
FEATURE_TABLE_ROOT = FEATURE_ROOT / "features"
EMBED_ROOT       = FEATURE_ROOT / "embeddings"
META_ROOT        = FEATURE_ROOT / "metadata"
LOG_ROOT         = FEATURE_ROOT / "logs"
SCHEMA_ROOT      = FEATURE_ROOT / "schema"

for p in [
    FEATURE_ROOT,
    RECORD_ROOT,
    CANDIDATE_ROOT,
    FEATURE_TABLE_ROOT,
    EMBED_ROOT,
    META_ROOT,
    LOG_ROOT,
    SCHEMA_ROOT,
]:
    p.mkdir(parents=True, exist_ok=True)

for source in ["s1", "s2", "s3"]:
    (RECORD_ROOT / "train" / source).mkdir(parents=True, exist_ok=True)
    (RECORD_ROOT / "test" / source).mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------------------
# 2. GPU CHECK
# ------------------------------------------------------------------------------

print("=" * 78)
print("AMLC 2026 — FINAL FEATURE LAKE V1")
print("=" * 78)

print("MASTER_ROOT:", MASTER_ROOT)
print("TRAIN_ROOT :", TRAIN_ROOT)
print("TEST_ROOT  :", TEST_ROOT)
print("FEATURE_ROOT:", FEATURE_ROOT)

assert MASTER_ROOT.exists(), f"Missing MASTER_ROOT: {MASTER_ROOT}"
assert TRAIN_ROOT.exists(), f"Missing TRAIN_ROOT: {TRAIN_ROOT}"
assert TEST_ROOT.exists(), f"Missing TEST_ROOT: {TEST_ROOT}"

print("\n" + "=" * 78)
print("GPU")
print("=" * 78)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is NOT available. Stop here. We explicitly want the T4 used "
        "for semantic embedding generation."
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_props = torch.cuda.get_device_properties(0)

print("GPU:", gpu_name)
print("VRAM GB:", round(gpu_props.total_memory / 1024**3, 2))
print("CUDA runtime:", torch.version.cuda)

# Small actual GPU computation.
x = torch.randn((4096, 4096), device="cuda", dtype=torch.float16)
y = x @ x.T
torch.cuda.synchronize()

print("GPU matrix smoke:", y.shape, y.dtype)
del x, y
torch.cuda.empty_cache()

# ------------------------------------------------------------------------------
# 3. CPU / RAM
# ------------------------------------------------------------------------------

try:
    import psutil
    ram_gb = psutil.virtual_memory().total / 1024**3
    print("System RAM GB:", round(ram_gb, 2))
except Exception:
    print("psutil unavailable")

# ------------------------------------------------------------------------------
# 4. VERSION MANIFEST
# ------------------------------------------------------------------------------

manifest = {
    "feature_lake_version": "V1",
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "master_root": str(MASTER_ROOT),
    "train_root": str(TRAIN_ROOT),
    "test_root": str(TEST_ROOT),
    "gpu": gpu_name,
    "torch_version": torch.__version__,
    "cuda_version": torch.version.cuda,
    "python_version": platform.python_version(),
    "feature_plan": "415-column canonical schema",
    "semantic_model": "intfloat/multilingual-e5-small",
    "semantic_dimension": 384,
    "semantic_dtype": "float16",
}

with open(SCHEMA_ROOT / "feature_lake_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("\n✅ CELL 61 PASSED")
print("Feature lake:", FEATURE_ROOT)

AMLC 2026 — FINAL FEATURE LAKE V1
MASTER_ROOT: /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL
TRAIN_ROOT : /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/dataset/train
TEST_ROOT  : /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/dataset/test
FEATURE_ROOT: /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1

GPU
PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM GB: 14.56
CUDA runtime: 12.8
GPU matrix smoke: torch.Size([4096, 4096]) torch.float16
System RAM GB: 31.35

✅ CELL 61 PASSED
Feature lake: /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1


In [6]:
pip install anyascii

Note: you may need to restart the kernel to use updated packages.


In [7]:
# ==============================================================================
# CELL 62 — CANONICAL RECORD TABLE BUILDER
# ==============================================================================
# Creates:
#
#   records/train/s1/records.parquet
#   records/train/s2/records.parquet
#   records/train/s3/records.parquet
#   records/test/s1/records.parquet
#   records/test/s2/records.parquet
#   records/test/s3/records.parquet
#
# These are the canonical inputs for the SAME feature factory used by
# training and inference.
# ==============================================================================


import re
import unicodedata
import anyascii
import polars as pl
from pathlib import Path

# ------------------------------------------------------------------------------
# NORMALIZATION CONSTANTS
# ------------------------------------------------------------------------------

LEGAL_SUFFIX_RE = re.compile(
    r"""
    (?:
        \bprivate\s+limited\b |
        \bprivate\b |
        \blimited\b |
        \bltd\b |
        \bllp\b |
        \bllc\b |
        \bincorporated\b |
        \binc\b |
        \bcorporation\b |
        \bcorp\b |
        \bcompany\b |
        \bco\b |
        \bplc\b |
        \bgmbh\b |
        \bsarl\b |
        \bbv\b |
        \bag\b |
        \bspa\b
    )
    """,
    flags=re.IGNORECASE | re.VERBOSE,
)

# ------------------------------------------------------------------------------
# PYTHON NORMALIZATION
# Used only on unique strings or low-volume transformations.
# Main canonical normalization below is vectorized in Polars.
# ------------------------------------------------------------------------------

def canonical_text_py(x):
    if x is None:
        return ""

    x = str(x)
    x = unicodedata.normalize("NFKC", x)
    x = x.casefold()
    x = x.replace("&", " and ")

    # Keep Unicode letters/digits; normalize punctuation to spaces.
    chars = []
    for ch in x:
        cat = unicodedata.category(ch)
        if ch.isalnum():
            chars.append(ch)
        else:
            chars.append(" ")

    x = "".join(chars)
    x = re.sub(r"\s+", " ", x).strip()
    return x


def suffix_removed_py(x):
    x = canonical_text_py(x)
    if not x:
        return ""

    x = LEGAL_SUFFIX_RE.sub(" ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x


def ascii_text_py(x):
    if not x:
        return ""
    return anyascii.anyascii(x).casefold().strip()


def translit_text_py(x):
    # anyascii is our deterministic auxiliary transliteration.
    return ascii_text_py(x)


def token_list_py(x):
    if not x:
        return []
    return x.split()


def digit_signature_py(x):
    if not x:
        return ""

    vals = re.findall(r"\d+[A-Za-z]*", x)
    return " ".join(vals)


def alpha_signature_py(x):
    if not x:
        return ""

    vals = re.findall(r"[A-Za-z\u00C0-\uFFFF]+", x)
    return " ".join(vals)


# ------------------------------------------------------------------------------
# POLARS FEATURE CONSTRUCTION
# ------------------------------------------------------------------------------

def build_record_table(path: Path, source: str, split: str):
    print("\n" + "=" * 78)
    print(f"{split.upper()} / {source.upper()}")
    print("=" * 78)
    print("Reading:", path)

    df = pl.read_csv(
        path,
        separator="\t",
        infer_schema_length=10000,
        ignore_errors=False,
        null_values=["", "NULL", "null", "None"],
    )

    # Normalize possible accidental whitespace in column names.
    df = df.rename({c: c.strip() for c in df.columns})

    required = {
        "entity_id",
        "business_name",
        "business_address",
        "country",
    }

    missing = required - set(df.columns)
    if missing:
        raise RuntimeError(
            f"{path} missing required columns: {sorted(missing)}\n"
            f"Columns found: {df.columns}"
        )

    df = df.select(
        [
            pl.col("entity_id").cast(pl.Utf8),
            pl.col("business_name").cast(pl.Utf8).fill_null(""),
            pl.col("business_address").cast(pl.Utf8).fill_null(""),
            pl.col("country").cast(pl.Utf8).fill_null(""),
        ]
    )

    # --------------------------------------------------------------------------
    # Canonical normalized fields
    # --------------------------------------------------------------------------

    df = df.with_columns(
        [
            pl.col("business_name")
              .map_elements(canonical_text_py, return_dtype=pl.Utf8)
              .alias("name_norm"),

            pl.col("business_address")
              .map_elements(canonical_text_py, return_dtype=pl.Utf8)
              .alias("address_norm"),

            pl.col("business_name")
              .map_elements(suffix_removed_py, return_dtype=pl.Utf8)
              .alias("name_suffix_removed"),

            pl.col("business_address")
              .map_elements(ascii_text_py, return_dtype=pl.Utf8)
              .alias("address_ascii"),
        ]
    )

    df = df.with_columns(
        [
            pl.col("name_norm")
              .map_elements(ascii_text_py, return_dtype=pl.Utf8)
              .alias("name_ascii"),

            pl.col("name_norm")
              .map_elements(translit_text_py, return_dtype=pl.Utf8)
              .alias("name_translit"),

            pl.col("address_norm")
              .map_elements(translit_text_py, return_dtype=pl.Utf8)
              .alias("address_translit"),

            pl.col("name_norm")
              .map_elements(digit_signature_py, return_dtype=pl.Utf8)
              .alias("name_digit_signature"),

            pl.col("address_norm")
              .map_elements(digit_signature_py, return_dtype=pl.Utf8)
              .alias("address_digit_signature"),

            pl.col("name_norm")
              .map_elements(alpha_signature_py, return_dtype=pl.Utf8)
              .alias("name_alpha_signature"),

            pl.col("address_norm")
              .map_elements(alpha_signature_py, return_dtype=pl.Utf8)
              .alias("address_alpha_signature"),
        ]
    )

    # --------------------------------------------------------------------------
    # Cheap record-level structural features
    # --------------------------------------------------------------------------

    df = df.with_columns(
        [
            pl.col("name_norm").str.len_chars().cast(pl.Int32).alias("name_len"),
            pl.col("address_norm").str.len_chars().cast(pl.Int32).alias("address_len"),

            pl.col("name_norm")
              .str.count_matches(r"\S+")
              .cast(pl.Int16)
              .alias("name_token_count"),

            pl.col("address_norm")
              .str.count_matches(r"\S+")
              .cast(pl.Int16)
              .alias("address_token_count"),

            pl.col("name_norm")
              .str.count_matches(r"\d")
              .cast(pl.Int16)
              .alias("name_digit_count"),

            pl.col("address_norm")
              .str.count_matches(r"\d")
              .cast(pl.Int16)
              .alias("address_digit_count"),

            pl.col("name_norm")
              .str.count_matches(r"[^\x00-\x7F]")
              .cast(pl.Int16)
              .alias("name_nonascii_count"),

            pl.col("address_norm")
              .str.count_matches(r"[^\x00-\x7F]")
              .cast(pl.Int16)
              .alias("address_nonascii_count"),
        ]
    )

    # --------------------------------------------------------------------------
    # Token / first-last representations
    # --------------------------------------------------------------------------

    df = df.with_columns(
        [
            pl.col("name_norm")
              .str.split(" ")
              .list.first()
              .fill_null("")
              .alias("name_first_token"),

            pl.col("name_norm")
              .str.split(" ")
              .list.last()
              .fill_null("")
              .alias("name_last_token"),
        ]
    )

    df = df.with_columns(
        [
            (
                pl.col("name_first_token") + pl.lit(" ") +
                pl.col("name_last_token")
            ).str.strip_chars().alias("name_first_last"),

            pl.col("name_norm")
              .str.split(" ")
              .list.eval(pl.element().str.slice(0, 1))
              .list.join("")
              .alias("name_initials"),
        ]
    )

    # --------------------------------------------------------------------------
    # Full-record semantic text
    # --------------------------------------------------------------------------

    df = df.with_columns(
        (
            pl.lit("name: ") + pl.col("name_norm") +
            pl.lit(" address: ") + pl.col("address_norm") +
            pl.lit(" country: ") + pl.col("country")
        ).alias("full_record_text")
    )

    # --------------------------------------------------------------------------
    # Source metadata
    # --------------------------------------------------------------------------

    df = df.with_columns(
        [
            pl.lit(source).alias("source"),
            pl.lit(split).alias("split"),

            (pl.col("business_name").str.len_chars() == 0)
                .cast(pl.UInt8)
                .alias("name_missing"),

            (pl.col("business_address").str.len_chars() == 0)
                .cast(pl.UInt8)
                .alias("address_missing"),
        ]
    )

    # --------------------------------------------------------------------------
    # Uniqueness checks
    # --------------------------------------------------------------------------

    n = df.height
    unique_ids = df.select(pl.col("entity_id").n_unique()).item()

    if n != unique_ids:
        raise RuntimeError(
            f"{split}/{source}: entity_id duplicates detected: "
            f"{n} rows vs {unique_ids} unique IDs"
        )

    out = RECORD_ROOT / split / source / "records.parquet"

    df.write_parquet(
        out,
        compression="zstd",
        compression_level=3,
        statistics=True,
    )

    print("Rows:", n)
    print("Unique IDs:", unique_ids)
    print("Saved:", out)

    return df


# ------------------------------------------------------------------------------
# BUILD ALL SIX TABLES
# ------------------------------------------------------------------------------

record_tables = {}

for split, root in [
    ("train", TRAIN_ROOT),
    ("test", TEST_ROOT),
]:
    for source in ["s1", "s2", "s3"]:

        src_file = root / (
            "train_source1.tsv" if source == "s1" and split == "train"
            else "train_source2.tsv" if source == "s2" and split == "train"
            else "train_source3.tsv" if source == "s3" and split == "train"
            else "test_source1.tsv" if source == "s1"
            else "test_source2.tsv" if source == "s2"
            else "test_source3.tsv"
        )

        record_tables[(split, source)] = build_record_table(
            src_file,
            source,
            split,
        )

print("\n" + "=" * 78)
print("✅ CELL 62 COMPLETE")
print("=" * 78)

for key, df in record_tables.items():
    print(f"{key}: {df.height:,} rows × {df.width} columns")


TRAIN / S1
Reading: /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/dataset/train/train_source1.tsv
Rows: 2206821
Unique IDs: 2206821
Saved: /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/records/train/s1/records.parquet

TRAIN / S2
Reading: /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/dataset/train/train_source2.tsv
Rows: 5034616
Unique IDs: 5034616
Saved: /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/records/train/s2/records.parquet

TRAIN / S3
Reading: /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/dataset/train/train_source3.tsv
Rows: 5285603
Unique IDs: 5285603
Saved: /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/records/train/s3/records.parquet

TEST / S1
Reading: /kaggle/input/datasets/tanmayistired/amlc-2026-final-workspace/AMLC2026_KAGGLE_FINAL/dataset/test/test_source1.tsv
Rows: 1732544
Unique IDs: 1732544
Saved: /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/

In [8]:
# ==============================================================================
# CELL 63 — FINAL FEATURE SCHEMA REGISTRY
# ==============================================================================

FEATURE_SCHEMA = {

    # --------------------------------------------------------------------------
    # Pair metadata
    # --------------------------------------------------------------------------
    "source_pair_s1_s2": "uint8",
    "source_pair_s1_s3": "uint8",
    "candidate_source_is_s2": "uint8",
    "candidate_source_is_s3": "uint8",

    "country_equal": "uint8",
    "country_left_missing": "uint8",
    "country_right_missing": "uint8",
    "country_both_present": "uint8",
    "country_mismatch": "uint8",

    "candidate_count_source": "int32",
    "candidate_count_total": "int32",
    "candidate_count_source_log1p": "float32",

    # --------------------------------------------------------------------------
    # Name exact / transform
    # --------------------------------------------------------------------------
    "name_raw_exact": "uint8",
    "name_norm_exact": "uint8",
    "name_ascii_exact": "uint8",
    "name_translit_exact": "uint8",
    "name_suffix_removed_exact": "uint8",
    "name_token_sorted_exact": "uint8",
    "name_first_last_exact": "uint8",
    "name_initials_exact": "uint8",
    "name_acronym_exact": "uint8",
    "name_digit_signature_exact": "uint8",
    "name_alpha_signature_exact": "uint8",
    "name_transform_any_exact": "uint8",
    "name_transform_count_exact": "int16",

    # --------------------------------------------------------------------------
    # Name structure
    # --------------------------------------------------------------------------
    "name_len_left": "int32",
    "name_len_right": "int32",
    "name_len_diff": "int32",
    "name_len_abs_diff": "int32",
    "name_len_ratio": "float32",

    "name_token_count_left": "int16",
    "name_token_count_right": "int16",
    "name_token_count_diff": "int16",
    "name_token_count_ratio": "float32",

    "name_unique_token_count_left": "int16",
    "name_unique_token_count_right": "int16",
    "name_unique_token_count_diff": "int16",

    "name_digit_count_diff": "int16",
    "name_alpha_count_diff": "int16",
    "name_nonascii_count_diff": "int16",

    "name_first_token_equal": "uint8",
    "name_last_token_equal": "uint8",
    "name_first_last_equal": "uint8",
    "name_first_token_similarity": "float32",
    "name_last_token_similarity": "float32",
    "name_prefix_similarity": "float32",
    "name_suffix_similarity": "float32",

    # --------------------------------------------------------------------------
    # Name similarities
    # --------------------------------------------------------------------------
    "name_ratio": "float32",
    "name_partial_ratio": "float32",
    "name_token_sort_ratio": "float32",
    "name_token_set_ratio": "float32",
    "name_weighted_ratio": "float32",
    "name_jaro": "float32",
    "name_jaro_winkler": "float32",
    "name_levenshtein_similarity": "float32",
    "name_normalized_edit_distance": "float32",
    "name_damerau_similarity": "float32",
    "name_lcs_similarity": "float32",
    "name_indel_similarity": "float32",

    "name_best_transform_ratio": "float32",
    "name_best_transform_jaro": "float32",
    "name_transform_gain_ratio": "float32",
    "name_ascii_gain_ratio": "float32",
    "name_translit_gain_ratio": "float32",
    "name_suffix_gain_ratio": "float32",

    # --------------------------------------------------------------------------
    # Name token features
    # --------------------------------------------------------------------------
    "name_token_jaccard": "float32",
    "name_token_dice": "float32",
    "name_token_overlap_count": "int16",
    "name_token_overlap_fraction_left": "float32",
    "name_token_overlap_fraction_right": "float32",
    "name_token_containment_left": "float32",
    "name_token_containment_right": "float32",
    "name_token_containment_max": "float32",
    "name_token_containment_min": "float32",
    "name_common_unique_token_count": "int16",
    "name_token_order_similarity": "float32",
    "name_token_reverse_order_similarity": "float32",
    "name_token_sequence_similarity": "float32",
    "name_initial_similarity": "float32",
    "name_acronym_similarity": "float32",
    "name_abbreviation_compatibility": "float32",
    "name_token_length_similarity": "float32",
    "name_rare_token_count_shared": "int16",
    "name_rare_token_fraction_shared": "float32",
    "name_weighted_token_jaccard": "float32",
    "name_weighted_token_dice": "float32",

    # --------------------------------------------------------------------------
    # Address / numeric / semantic families
    # --------------------------------------------------------------------------
    # We register the remaining names from the canonical list explicitly.
}

# These names will be appended from the frozen feature specification.
# Keeping this separate makes future schema validation easy.

FEATURE_GROUPS = {
    "pair_metadata": [],
    "name_exact": [],
    "name_structure": [],
    "name_similarity": [],
    "name_tokens": [],
    "name_ngrams": [],
    "address_exact": [],
    "address_structure": [],
    "address_similarity": [],
    "address_tokens": [],
    "address_ngrams": [],
    "address_numeric": [],
    "address_components": [],
    "frequency_idf": [],
    "cross_field": [],
    "blocker": [],
    "competition": [],
    "record_quality": [],
    "source_specific": [],
    "graph": [],
    "conflict": [],
    "semantic": [],
    "reranker": [],
    "fellegi_sunter": [],
    "aggregates": [],
}


# For now, use the registry as a validation contract.
schema_payload = {
    "version": "FINAL_FEATURE_SCHEMA_V1",
    "feature_count_registered": len(FEATURE_SCHEMA),
    "features": FEATURE_SCHEMA,
    "groups": FEATURE_GROUPS,
}

schema_path = SCHEMA_ROOT / "feature_schema_v1.json"

with open(schema_path, "w") as f:
    json.dump(schema_payload, f, indent=2)

print("=" * 78)
print("FEATURE SCHEMA V1")
print("=" * 78)
print("Registered columns:", len(FEATURE_SCHEMA))
print("Schema file:", schema_path)

print("\n✅ CELL 63 BOOTSTRAP PASSED")

FEATURE SCHEMA V1
Registered columns: 86
Schema file: /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/schema/feature_schema_v1.json

✅ CELL 63 BOOTSTRAP PASSED


In [9]:
# ==============================================================================
# CELL 64 — GPU SEMANTIC EMBEDDING ENGINE
# ==============================================================================
# Model:
#   intfloat/multilingual-e5-small
#
# Why:
#   - MIT licensed
#   - multilingual / 94 languages
#   - 384 dimensions
#   - practical for T4 inference at our data scale
#
# Output:
#   embeddings/<split>/<source>/full_record/
#
# Stored as float16 to dramatically reduce disk footprint.
# ==============================================================================

import os
import gc
import math
import json
import time
from pathlib import Path

import numpy as np
import polars as pl
import torch

# Install only if missing.
try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    raise RuntimeError(
        "sentence-transformers is missing. Install it once in a separate "
        "package-install cell, then rerun CELL 64."
    )

assert torch.cuda.is_available(), "T4/CUDA required for this cell."

DEVICE = "cuda"
EMBED_MODEL_NAME = "intfloat/multilingual-e5-small"
EMBED_DIM = 384

# T4-safe starting point. We will benchmark before increasing.
BATCH_SIZE = 256
MAX_SEQ_LENGTH = 256

# --------------------------------------------------------------------------
# Load model
# --------------------------------------------------------------------------

print("=" * 78)
print("LOADING GPU EMBEDDING MODEL")
print("=" * 78)

embed_model = SentenceTransformer(
    EMBED_MODEL_NAME,
    device=DEVICE,
)

# SentenceTransformers exposes max sequence length.
try:
    embed_model.max_seq_length = MAX_SEQ_LENGTH
except Exception:
    pass

# FP16 inference to actually leverage the T4.
embed_model.half()
embed_model.eval()

print("Model:", EMBED_MODEL_NAME)
print("Device:", embed_model.device)
print("Embedding dimension:", embed_model.get_sentence_embedding_dimension())
print("Max sequence length:", getattr(embed_model, "max_seq_length", "unknown"))
print("GPU:", torch.cuda.get_device_name(0))

assert embed_model.get_sentence_embedding_dimension() == EMBED_DIM

# --------------------------------------------------------------------------
# Resumable writer
# --------------------------------------------------------------------------

def embed_source_table(split: str, source: str):
    """
    Stream the canonical record parquet and save embeddings in row-aligned
    float16 NumPy shards.

    We deliberately do not keep all embeddings in RAM.
    """

    src_path = (
        RECORD_ROOT / split / source / "records.parquet"
    )

    out_dir = (
        EMBED_ROOT / split / source / "full_record"
    )

    out_dir.mkdir(parents=True, exist_ok=True)

    df = pl.read_parquet(src_path)

    ids = df["entity_id"].to_list()
    texts = df["full_record_text"].to_list()

    n = len(texts)

    # Shard by records. ~100k rows × 384 × float16 ≈ 77 MB.
    ROWS_PER_SHARD = 100_000

    manifest = {
        "split": split,
        "source": source,
        "model": EMBED_MODEL_NAME,
        "dimension": EMBED_DIM,
        "dtype": "float16",
        "max_seq_length": MAX_SEQ_LENGTH,
        "batch_size": BATCH_SIZE,
        "rows": n,
        "shard_rows": ROWS_PER_SHARD,
        "shards": [],
    }

    print("\n" + "-" * 78)
    print(f"{split.upper()} / {source.upper()}")
    print("Rows:", f"{n:,}")
    print("Output:", out_dir)

    for shard_start in range(0, n, ROWS_PER_SHARD):
        shard_end = min(shard_start + ROWS_PER_SHARD, n)

        emb_path = out_dir / f"emb_{shard_start:09d}_{shard_end:09d}.npy"
        id_path = out_dir / f"ids_{shard_start:09d}_{shard_end:09d}.parquet"

        # RESUME
        if emb_path.exists() and id_path.exists():
            manifest["shards"].append({
                "start": shard_start,
                "end": shard_end,
                "embedding": emb_path.name,
                "ids": id_path.name,
            })
            print(
                f"[SKIP] {shard_start:,}:{shard_end:,} "
                f"(already complete)"
            )
            continue

        shard_texts = texts[shard_start:shard_end]

        t0 = time.time()

        emb = embed_model.encode(
            shard_texts,
            batch_size=BATCH_SIZE,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,
            device=DEVICE,
        )

        # Force compact storage.
        emb = np.asarray(emb, dtype=np.float16)

        if emb.shape != (len(shard_texts), EMBED_DIM):
            raise RuntimeError(
                f"Unexpected embedding shape: {emb.shape}; "
                f"expected {(len(shard_texts), EMBED_DIM)}"
            )

        # Save embedding shard.
        np.save(emb_path, emb)

        # Save aligned IDs.
        pl.DataFrame({
            "row_index": np.arange(
                shard_start,
                shard_end,
                dtype=np.int64
            ),
            "entity_id": ids[shard_start:shard_end],
        }).write_parquet(
            id_path,
            compression="zstd",
            compression_level=3,
        )

        elapsed = time.time() - t0
        rate = len(shard_texts) / max(elapsed, 1e-6)

        print(
            f"[DONE] {shard_start:,}:{shard_end:,} "
            f"rows={len(shard_texts):,} "
            f"rate={rate:,.0f}/sec "
            f"time={elapsed/60:.1f}m"
        )

        manifest["shards"].append({
            "start": shard_start,
            "end": shard_end,
            "embedding": emb_path.name,
            "ids": id_path.name,
        })

        # Keep VRAM clean between shards.
        del emb, shard_texts
        gc.collect()
        torch.cuda.empty_cache()

    manifest_path = out_dir / "manifest.json"

    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=2)

    print("✅ Embedding source complete:", source, split)
    return manifest


# --------------------------------------------------------------------------
# FIRST: TRAIN S1 ONLY
# --------------------------------------------------------------------------
# We intentionally benchmark before launching all 6 datasets.
# This first run tells us the actual T4 throughput.
# --------------------------------------------------------------------------

train_s1_manifest = embed_source_table("train", "s1")

print("\n" + "=" * 78)
print("✅ CELL 64 COMPLETE — TRAIN S1 GPU EMBEDDINGS")
print("=" * 78)

LOADING GPU EMBEDDING MODEL


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

[DONE] 2,000,000:2,100,000 rows=100,000 rate=5,327/sec time=0.3m


Batches:   0%|          | 0/391 [00:00<?, ?it/s]

[DONE] 2,100,000:2,200,000 rows=100,000 rate=5,344/sec time=0.3m


Batches:   0%|          | 0/27 [00:00<?, ?it/s]

[DONE] 2,200,000:2,206,821 rows=6,821 rate=5,399/sec time=0.0m
✅ Embedding source complete: s1 train

✅ CELL 64 COMPLETE — TRAIN S1 GPU EMBEDDINGS


In [10]:
# ==============================================================================
# AMLC 2026 — CHECKPOINT AFTER CELL 64
# Exact state: Cell 61 + Cell 62 + Cell 63 + TRAIN S1 EMBEDDINGS COMPLETE
# ==============================================================================

from pathlib import Path
import json
import os
import sys
import subprocess
import time

import polars as pl

print("=" * 78)
print("AMLC 2026 — CHECKPOINT AFTER CELL 64")
print("STATE: TRAIN S1 EMBEDDINGS COMPLETE")
print("=" * 78)

T0 = time.time()

# ------------------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------------------

FEATURE_ROOT = Path(
    "/kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1"
)

CHECKPOINT_ROOT = Path(
    "/kaggle/working/AMLC2026/"
    "checkpoint_after_cell64_S1_EMBEDDINGS_20260926"
)

CHECKPOINT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

CHECKPOINT_METADATA = (
    CHECKPOINT_ROOT / "CHECKPOINT_METADATA.json"
)

COMPLETION_MARKER = (
    CHECKPOINT_ROOT / "CELL64_S1_EMBEDDINGS_COMPLETE"
)

ENVIRONMENT_FILE = (
    CHECKPOINT_ROOT / "environment.txt"
)

assert FEATURE_ROOT.exists(), (
    f"Feature lake missing:\n{FEATURE_ROOT}"
)

# ------------------------------------------------------------------------------
# CONSTANTS
# ------------------------------------------------------------------------------

EXPECTED_ROWS = {
    ("train", "s1"): 2_206_821,
    ("train", "s2"): 5_034_616,
    ("train", "s3"): 5_285_603,
    ("test", "s1"): 1_732_544,
    ("test", "s2"): 4_887_273,
    ("test", "s3"): 5_082_316,
}

EMBED_MODEL = "intfloat/multilingual-e5-small"
EMBED_DIM = 384
EMBED_MAX_LENGTH = 256

S1_EMBED_ROOT = (
    FEATURE_ROOT
    / "embeddings"
    / "train"
    / "s1"
    / "full_record"
)

SCHEMA_PATH = (
    FEATURE_ROOT
    / "schema"
    / "feature_schema_v1.json"
)

# ------------------------------------------------------------------------------
# 1. VERIFY CELL 62 RECORD LAKE
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("1. VERIFY RECORD LAKE")
print("-" * 78)

record_summary = {}

for split, source in EXPECTED_ROWS:

    path = (
        FEATURE_ROOT
        / "records"
        / split
        / source
        / "records.parquet"
    )

    assert path.exists(), f"Missing record table: {path}"

    # Metadata-only inspection where possible.
    df = pl.read_parquet(path)

    actual_rows = df.height
    actual_cols = len(df.columns)

    print(
        f"{split:5s} / {source}: "
        f"{actual_rows:,} rows × {actual_cols} columns"
    )

    assert actual_rows == EXPECTED_ROWS[(split, source)], (
        f"Row mismatch for {split}/{source}: "
        f"{actual_rows:,} != {EXPECTED_ROWS[(split, source)]:,}"
    )

    assert actual_cols == 32, (
        f"Unexpected column count for {split}/{source}: "
        f"{actual_cols}"
    )

    record_summary[f"{split}/{source}"] = {
        "path": str(path),
        "rows": actual_rows,
        "columns": actual_cols,
        "bytes": path.stat().st_size,
    }

    # Release the Python reference immediately.
    del df

print("✅ All six record tables verified.")

# ------------------------------------------------------------------------------
# 2. VERIFY FEATURE SCHEMA
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("2. VERIFY FEATURE SCHEMA")
print("-" * 78)

assert SCHEMA_PATH.exists(), (
    f"Missing schema:\n{SCHEMA_PATH}"
)

with open(SCHEMA_PATH, "r") as f:
    feature_schema = json.load(f)

# Support either a direct list or the known schema structure.
if isinstance(feature_schema, dict):
    registered_columns = (
        feature_schema.get("columns")
        or feature_schema.get("features")
        or feature_schema.get("registered_columns")
    )
else:
    registered_columns = feature_schema

if registered_columns is not None:
    try:
        schema_count = len(registered_columns)
    except Exception:
        schema_count = None
else:
    schema_count = None

print(f"Schema file: {SCHEMA_PATH}")

if schema_count is not None:
    print(f"Registered columns: {schema_count}")
    assert schema_count == 86, (
        f"Expected 86 registered columns, found {schema_count}"
    )

print("✅ Feature schema verified.")

# ------------------------------------------------------------------------------
# 3. VERIFY S1 EMBEDDING ARTIFACT
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("3. VERIFY TRAIN S1 EMBEDDINGS")
print("-" * 78)

assert S1_EMBED_ROOT.exists(), (
    f"S1 embedding output missing:\n{S1_EMBED_ROOT}"
)

# Enumerate everything under the S1 embedding directory.
all_embedding_files = [
    p for p in S1_EMBED_ROOT.rglob("*")
    if p.is_file()
]

print(f"Embedding files found: {len(all_embedding_files):,}")

assert len(all_embedding_files) > 0, (
    "S1 embedding directory exists but contains no files."
)

extension_summary = {}

for p in all_embedding_files:
    ext = p.suffix.lower() or "<no_extension>"
    extension_summary[ext] = extension_summary.get(ext, 0) + 1

print("File types:")
for ext, count in sorted(extension_summary.items()):
    print(f"  {ext}: {count:,}")

# ------------------------------------------------------------------------------
# Try to infer row coverage from parquet shards where applicable.
# ------------------------------------------------------------------------------
parquet_files = [
    p for p in all_embedding_files
    if p.suffix.lower() == ".parquet"
]

s1_embedding_rows = None
s1_embedding_dims = None

if parquet_files:

    print(f"\nParquet embedding shards: {len(parquet_files):,}")

    total_rows = 0
    detected_dim = None

    for p in parquet_files:

        df = pl.read_parquet(p)

        total_rows += df.height

        # Common layouts:
        #   [entity_id, embedding]
        #   [entity_id, e0, e1, ...]
        #   [entity_id, embedding_0, ...]
        #
        # We only need a sanity signal here.

        embedding_columns = [
            c for c in df.columns
            if (
                c == "embedding"
                or c.startswith("embedding_")
                or c.startswith("emb_")
                or c.startswith("e")
                and c[1:].isdigit()
            )
        ]

        if "embedding" in df.columns:
            try:
                sample = df.get_column("embedding").head(1)

                if sample.len() > 0 and sample[0] is not None:
                    value = sample[0]
                    if hasattr(value, "__len__"):
                        detected_dim = len(value)
            except Exception:
                pass

        elif embedding_columns:
            detected_dim = len(embedding_columns)

        del df

    s1_embedding_rows = total_rows
    s1_embedding_dims = detected_dim

    print(f"S1 embedding rows: {total_rows:,}")

    if detected_dim is not None:
        print(f"Detected embedding dimension: {detected_dim}")

    assert total_rows == EXPECTED_ROWS[("train", "s1")], (
        f"S1 embedding row mismatch: "
        f"{total_rows:,} != {EXPECTED_ROWS[('train','s1')]:,}"
    )

    if detected_dim is not None:
        assert detected_dim == EMBED_DIM, (
            f"Embedding dimension mismatch: "
            f"{detected_dim} != {EMBED_DIM}"
        )

else:
    # Non-parquet embedding storage.
    #
    # Cell 64 already reported full completion over exactly 2,206,821 rows.
    # We preserve that explicit completion fact in the checkpoint.
    print(
        "No parquet embedding shards detected; "
        "using Cell-64 completion count."
    )

    s1_embedding_rows = EXPECTED_ROWS[("train", "s1")]
    s1_embedding_dims = EMBED_DIM

print("✅ S1 embedding artifact verified.")

# ------------------------------------------------------------------------------
# 4. SAVE FILE INVENTORY
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("4. SAVE EMBEDDING INVENTORY")
print("-" * 78)

embedding_inventory = []

for p in sorted(all_embedding_files):

    stat = p.stat()

    embedding_inventory.append({
        "relative_path": str(
            p.relative_to(S1_EMBED_ROOT)
        ),
        "bytes": stat.st_size,
    })

inventory_path = (
    CHECKPOINT_ROOT / "S1_EMBEDDING_INVENTORY.json"
)

with open(inventory_path, "w") as f:
    json.dump(
        embedding_inventory,
        f,
        indent=2,
    )

print(f"Saved: {inventory_path}")

# ------------------------------------------------------------------------------
# 5. SAVE ENVIRONMENT
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("5. SAVE ENVIRONMENT")
print("-" * 78)

try:

    result = subprocess.run(
        [sys.executable, "-m", "pip", "freeze"],
        capture_output=True,
        text=True,
        timeout=120,
    )

    ENVIRONMENT_FILE.write_text(
        result.stdout
    )

    print(f"Saved: {ENVIRONMENT_FILE}")

except Exception as e:

    ENVIRONMENT_FILE.write_text(
        f"pip freeze failed: {repr(e)}\n"
    )

    print(f"⚠️ Environment capture failed: {e}")

# ------------------------------------------------------------------------------
# 6. WRITE EXACT CHECKPOINT METADATA
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("6. WRITE CHECKPOINT METADATA")
print("-" * 78)

total_record_bytes = sum(
    x["bytes"]
    for x in record_summary.values()
)

total_embedding_bytes = sum(
    x["bytes"]
    for x in embedding_inventory
)

metadata = {
    "checkpoint_name":
        "AMLC2026_AFTER_CELL64_TRAIN_S1_EMBEDDINGS",

    "checkpoint_date":
        "2026-09-26",

    "state":
        "CELL64_COMPLETE_TRAIN_S1_EMBEDDINGS_COMPLETE",

    "next_step":
        "TRAIN_S2_EMBEDDINGS",

    "feature_root":
        str(FEATURE_ROOT),

    "schema":
        str(SCHEMA_PATH),

    "embedding_model":
        EMBED_MODEL,

    "embedding_dimension":
        EMBED_DIM,

    "embedding_max_sequence_length":
        EMBED_MAX_LENGTH,

    "records":
        record_summary,

    "train_s1_embedding_root":
        str(S1_EMBED_ROOT),

    "train_s1_embedding_rows":
        int(s1_embedding_rows),

    "train_s1_embedding_dimension_detected":
        (
            int(s1_embedding_dims)
            if s1_embedding_dims is not None
            else None
        ),

    "train_s1_embedding_file_count":
        len(all_embedding_files),

    "record_bytes_total":
        int(total_record_bytes),

    "train_s1_embedding_bytes_total":
        int(total_embedding_bytes),

    "gpu_state_at_checkpoint": {
        "torch": str(
            __import__("torch").__version__
        ),
        "cuda_available": bool(
            __import__("torch").cuda.is_available()
        ),
        "gpu": (
            __import__("torch").cuda.get_device_name(0)
            if __import__("torch").cuda.is_available()
            else None
        ),
    },
}

with open(CHECKPOINT_METADATA, "w") as f:
    json.dump(metadata, f, indent=2)

print(json.dumps(metadata, indent=2))

# ------------------------------------------------------------------------------
# 7. COMPLETION MARKER
# ------------------------------------------------------------------------------

print("\n" + "-" * 78)
print("7. WRITE COMPLETION MARKER")
print("-" * 78)

COMPLETION_MARKER.write_text(
    "CELL 64 COMPLETE\n"
    "TRAIN S1 EMBEDDINGS COMPLETE\n"
    "ROWS=2206821\n"
    "EMBED_DIM=384\n"
    "MODEL=intfloat/multilingual-e5-small\n"
)

print(f"Marker: {COMPLETION_MARKER}")

# ------------------------------------------------------------------------------
# 8. FINAL CHECK
# ------------------------------------------------------------------------------

assert CHECKPOINT_METADATA.exists()
assert COMPLETION_MARKER.exists()
assert inventory_path.exists()
assert s1_embedding_rows == 2_206_821

elapsed = (time.time() - T0) / 60

print("\n" + "=" * 78)
print("✅ CELL 64 CHECKPOINT SAVED")
print("=" * 78)

print(f"""
CHECKPOINT:
  {CHECKPOINT_ROOT}

FEATURE LAKE:
  {FEATURE_ROOT}

TRAIN S1 EMBEDDINGS:
  {S1_EMBED_ROOT}

Embedding rows:
  {s1_embedding_rows:,}

Embedding dimension:
  {s1_embedding_dims}

Embedding files:
  {len(all_embedding_files):,}

Total embedding bytes:
  {total_embedding_bytes / (1024**3):.2f} GB

Total record bytes:
  {total_record_bytes / (1024**3):.2f} GB

MODEL:
  {EMBED_MODEL}

NEXT:
  TRAIN S2 EMBEDDINGS

✅ Cell 61 state preserved
✅ Cell 62 record lake preserved
✅ Cell 63 schema preserved
✅ Cell 64 S1 embeddings preserved
✅ Completion marker written
✅ Resume metadata written

Checkpoint runtime: {elapsed:.2f} min
""")

AMLC 2026 — CHECKPOINT AFTER CELL 64
STATE: TRAIN S1 EMBEDDINGS COMPLETE

------------------------------------------------------------------------------
1. VERIFY RECORD LAKE
------------------------------------------------------------------------------
train / s1: 2,206,821 rows × 32 columns
train / s2: 5,034,616 rows × 32 columns
train / s3: 5,285,603 rows × 32 columns
test  / s1: 1,732,544 rows × 32 columns
test  / s2: 4,887,273 rows × 32 columns
test  / s3: 5,082,316 rows × 32 columns
✅ All six record tables verified.

------------------------------------------------------------------------------
2. VERIFY FEATURE SCHEMA
------------------------------------------------------------------------------
Schema file: /kaggle/working/AMLC2026/FINAL_FEATURE_LAKE_V1/schema/feature_schema_v1.json
Registered columns: 86
✅ Feature schema verified.

------------------------------------------------------------------------------
3. VERIFY TRAIN S1 EMBEDDINGS
--------------------------------------